<a href="https://colab.research.google.com/github/Bersanone/Unet/blob/main/Copia_di_UNET_per_segmentazione.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import albumentations as A

# Dataset area flood segmentation

In [ ]:
os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""

In [ ]:
!kaggle datasets download faizalkarim/flood-area-segmentation

Dataset URL: https://www.kaggle.com/datasets/faizalkarim/flood-area-segmentation
License(s): CC0-1.0
  0% 0.00/107M [00:00<?, ?B/s]
100% 107M/107M [00:00<00:00, 1.76GB/s]


In [ ]:
!unzip /content/flood-area-segmentation.zip

In [ ]:
class FloodDataset(torch.utils.data.Dataset):
  def __init__(self):
    self.image_paths = sorted(os.listdir('/content/Image'))
    self.mask_paths = sorted(os.listdir('/content/Mask'))

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    image_path = os.path.join('/content/Image', self.image_paths[idx])
    mask_path = os.path.join('/content/Mask', self.mask_paths[idx])

    image = np.array(Image.open(image_path).convert('RGB'))
    image = image.astype(np.float32) / 255.0
    mask = np.array(Image.open(mask_path).convert('L'), dtype=np.float32)
    mask = np.where(mask > 127, 1.0, 0.0).astype(np.float32)

    # controllo dimensioni
    if image.shape[:2] != mask.shape or image.shape[2] != 3:
      print(image_path, mask_path)
      return None


    transform = A.Compose([
        A.Resize(128, 128),
    ])

    transformed = transform(image=image, mask=mask)
    image = transformed['image']
    mask = transformed['mask']

    image = image.transpose((2, 0, 1))

    image = torch.from_numpy(image)
    mask = torch.from_numpy(mask[None, ...])

    return image, mask

In [ ]:
ds = FloodDataset()

# Architettura Unet

In [ ]:
import torch
import torch.nn as nn

def get_model_sequential():
    return nn.Sequential(
        # --- ENCODER ---
        # Conv2D(64, 3, strides=2, padding="same") -> #2 nell'immagine
        nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1), # 64 x 64 x 64
        nn.ReLU(),

        nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), # 64 x 64 x 64
        nn.ReLU(),

        nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 32 x 32 x 128
        nn.ReLU(),

        nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1), # 32 x 32 x 128
        nn.ReLU(),

        nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1), # 16 x 16 x 256
        nn.ReLU(),

        nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1), # 16 x 16 x 256
        nn.ReLU(),

        nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1), # 8 x 8 x 256
        nn.ReLU(),

        # --- DECODER ---
        nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1), # 16 x 16 x 256
        nn.ReLU(),

        nn.ConvTranspose2d(256, 256, kernel_size=3, stride=1, padding=1), # 16 x 16 x 256
        nn.ReLU(),

        nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1), # 32 x 32 x 256
        nn.ReLU(),

        nn.ConvTranspose2d(256, 128, kernel_size=3, stride=1, padding=1), # 32 x 32 x 128
        nn.ReLU(),

        nn.ConvTranspose2d(128, 128, kernel_size=3, stride=2, padding=1, output_padding=1), # 64 x 64 x 128
        nn.ReLU(),

        nn.ConvTranspose2d(128, 64, kernel_size=3, stride=1, padding=1), # 64 x 64 x 64
        nn.ReLU(),

        nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1), # 128 x 128 x 64
        nn.ReLU(),

        # --- OUTPUT ---
        # Conv2D(num_classes, 3, activation="softmax", padding="same") -> #3 nell'immagine
        nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
        nn.Sigmoid()
    )

In [ ]:
model = get_model_sequential()

In [ ]:
out = model(ds[0][0][None, ...])

# Routine di addestramento

In [ ]:
dataloader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=True)

In [ ]:
LR = 1e-3
EPOCHS = 20

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

model.to("cuda")

for epoch in range(EPOCHS):
  for xbatch, ybatch in dataloader:
    xbatch = xbatch.to("cuda")
    ybatch = ybatch.to("cuda")
    out = model(xbatch)
    loss = criterion(out, ybatch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f'Epoch: {epoch+1}/{EPOCHS}, Loss: {loss.item()}')


Epoch: 1/20, Loss: 0.6933061480522156
Epoch: 1/20, Loss: 0.6918905973434448
Epoch: 1/20, Loss: 0.6903831362724304
Epoch: 1/20, Loss: 0.690491259098053
Epoch: 1/20, Loss: 0.6826061606407166
Epoch: 1/20, Loss: 0.7119409441947937
Epoch: 1/20, Loss: 0.680525541305542
Epoch: 1/20, Loss: 0.6906444430351257
Epoch: 1/20, Loss: 0.6825740933418274
Epoch: 2/20, Loss: 0.6865927577018738
Epoch: 2/20, Loss: 0.683090090751648
Epoch: 2/20, Loss: 0.6855407953262329
Epoch: 2/20, Loss: 0.6834187507629395
Epoch: 2/20, Loss: 0.6887573599815369
Epoch: 2/20, Loss: 0.686894416809082
Epoch: 2/20, Loss: 0.6736372709274292
Epoch: 2/20, Loss: 0.6945421695709229
Epoch: 2/20, Loss: 0.6707542538642883
Epoch: 3/20, Loss: 0.6879971027374268
Epoch: 3/20, Loss: 0.6666761040687561
Epoch: 3/20, Loss: 0.6910213232040405
Epoch: 3/20, Loss: 0.6665420532226562
Epoch: 3/20, Loss: 0.6677631139755249
Epoch: 3/20, Loss: 0.6866065263748169
Epoch: 3/20, Loss: 0.6515053510665894
Epoch: 3/20, Loss: 0.6583459377288818
Epoch: 3/20, Los

# Temp

In [ ]:
for i, _ in enumerate(ds):
  pass


In [ ]:
!rm /content/Image/1061.jpg /content/Mask/1061.png
!rm /content/Image/1079.jpg /content/Mask/1079.png
!rm /content/Image/14.jpg /content/Mask/14.png
!rm /content/Image/15.jpg /content/Mask/15.png
!rm /content/Image/2052.jpg /content/Mask/2052.png
!rm /content/Image/2053.jpg /content/Mask/2053.png
!rm /content/Image/3059.jpg /content/Mask/3059.png